# Phase folding and time binning

Use WASP-12 light curves to explore normalization, fixed-size time bins, and phase folding. The existing calculation folds at twice the reference period; retain that convention when interpreting the plot. Binning can suppress short-duration structure and does not create independent measurements.

## Before running
Run from the repository root after installing `requirements.txt`. Archive access requires internet. Inspect the Lightkurve search table and explicitly choose the intended sector/product: numeric result indices are not stable archive identifiers. Saved execution outputs are intentionally absent.

## Review the results
Compare raw and cleaned flux; inspect masked regions and model residuals. Record the selected data product, assumptions, and uncertainties before using a result in scientific work.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import lightkurve as lk
from scipy import optimize
from mandelagol import occultquad
from scipy.optimize import curve_fit

plt.rcParams['font.size'] = 24
plt.rcParams['figure.figsize'] = [20,10]

In [ ]:
result = lk.search_lightcurve('WASP-12', mission='TESS')
print(result)

In [ ]:
lc = result[3].download().remove_nans()

In [ ]:
time = lc.time.to_value('jd')
flux = lc.flux.to_value()
flux_norm = flux/np.median(flux)

In [ ]:
plt.figure(figsize=(16,8))
plt.plot(time,flux_norm,'.')
plt.xlabel('Time [BJD]')
plt.ylabel('Relative Flux')

In [ ]:
def time_average(time, flux, npts):
    
    # It is sometimes useful to averaging the data in time, to improve the signal-
    # to-noise ratio and reduce the data volume.
    
    # input: time, flux, and number of sequential data points that should be averaged
    # output: time_avg and flux_avg.
    
    # first make sure the data are sorted in time
    
    sorted = np.argsort(time)
    time_sorted = time[sorted]
    flux_sorted = flux[sorted]
    ndata = len(time_sorted)
    
    # now perform the averaging.
    
    ndata_avg = int(ndata/npts) # rounds downward to nearest integer

    time_avg = np.zeros(ndata_avg)
    flux_avg = np.zeros(ndata_avg)

    for i in range(ndata_avg):
        i_start = i*npts
        i_end = i_start + npts
        time_avg[i] = np.median(time_sorted[i_start:i_end])
        flux_avg[i] = np.median(flux_sorted[i_start:i_end])

    return(time_avg, flux_avg)

def fold_time(t, tc_ref, period_ref):
    
    '''
    
    Converts time into "folded time" in the range between -period/2 and +period/2
    as well as an orbit number, assigning tc_ref = orbit zero
     
    '''  
    
    orbit_number, t_fold = np.divmod( t - (tc_ref - 0.5*period_ref), period_ref )
    orbit_number = orbit_number.astype(int)
    
    t_fold = t_fold - 0.5*period_ref
 
    return(t_fold, orbit_number)


In [ ]:
period_ref = 1.091407 
tc_ref = 2.458E6 +  1510.95210 
time_fold, orbit = fold_time(time, tc_ref, 2*period_ref)

plt.plot(time_fold, flux_norm, 'o', alpha=0.1)
time_fold_avg, flux_avg = time_average(time_fold, flux_norm, 100)
plt.plot(time_fold_avg, flux_avg, 'ro')
#plt.ylim(0.995,1.005)
foo = plt.title('Time-averaged and phase-folded light curve')